In [ ]:
import traitlets
import ipywidgets.widgets as widgets
import time
import threading
import inspect
import ctypes
import cv2

origin_widget = widgets.Image(format='jpeg', width=320, height=240)
mask_widget = widgets.Image(format='jpeg',width=320, height=240)
result_widget = widgets.Image(format='jpeg',width=320, height=240)

image_container = widgets.HBox([origin_widget, mask_widget, result_widget])
# image_container = widgets.Image(format='jpeg', width=600, height=500)


In [ ]:
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg',value)[1])
    
# Initialize the widgets
origin_widget = widgets.Image(format='jpeg', width=320, height=240)
mask_widget = widgets.Image(format='jpeg', width=320, height=240)
result_widget = widgets.Image(format='jpeg', width=320, height=240)

# Create a horizontal container to place image widgets side-by-side
image_container = widgets.HBox([origin_widget, mask_widget, result_widget])

In [ ]:
def get_color(img):
    H = []
    color_name={}
    img = cv2.resize(img, (640, 480), )
  
    HSV = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
  
    cv2.rectangle(img, (280, 180), (360, 260), (0, 255, 0), 2)
    
    for i in range(280, 360):
        for j in range(180, 260): H.append(HSV[j, i][0])
  
    H_min = min(H);H_max = max(H)
#     print(H_min,H_max)
   
    if H_min >= 0 and H_max <= 10 or H_min >= 156 and H_max <= 180: color_name['name'] = 'red'
    elif H_min >= 26 and H_max <= 34: color_name['name'] = 'yellow'
    elif H_min >= 35 and H_max <= 78: color_name['name'] = 'green'
    elif H_min >= 100 and H_max <= 124: color_name['name'] = 'blue'
    return img, color_name

In [ ]:
import cv2
import numpy as np
import ipywidgets.widgets as widgets


cap = cv2.VideoCapture(0,cv2.CAP_V4L)
cap.set(3, 640)
cap.set(4, 480)
cap.set(5, 30)  
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M', 'J', 'P', 'G'))


color_lower = np.array([0, 43, 46])
color_upper = np.array([10, 255, 255])


def Color_Recongnize():
    m_fps = 0
    t_start = time.time()
    while(1):
        
        ret, frame = cap.read()
        frame, color_name = get_color(frame)
        if len(color_name)==1:
            print("color_name = ",color_name)
            global color_lower
            global color_upper

            if color_name['name'] == 'yellow':
                color_lower = np.array([12, 91, 235])
                color_upper = np.array([96, 156, 255])
                
            elif color_name['name'] == 'red':
                color_lower = np.array([0, 43, 46])
                color_upper = np.array([10, 255, 255])
               
            elif  color_name['name'] == 'green':
                color_lower = np.array([35, 43, 46])
                color_upper = np.array([77, 255, 255])
              
            elif color_name['name'] == 'blue':
                color_lower=np.array([100, 43, 46])
                color_upper = np.array([124, 255, 255])

        m_fps = m_fps + 1
        fps = m_fps / (time.time() - t_start)
        if (time.time() - t_start) >= 2:
            m_fps = fps
            t_start = time.time() - 1
        text = "FPS : " + str(int(fps))
        cv2.putText(frame, text, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 1)
        
        origin_widget.value = bgr8_to_jpeg(frame)
        #cv2.imshow('Capture', frame)

        # change to hsv model
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        
        mask = cv2.inRange(hsv, color_lower, color_upper)
        #cv2.imshow('Mask', mask)
        mask_widget.value = bgr8_to_jpeg(mask)


      
        res = cv2.bitwise_and(frame, frame, mask=mask)
        #cv2.imshow('Result', res)
        result_widget.value = bgr8_to_jpeg(res)

        #     if cv2.waitKey(A1) & 0xFF == ord('q'):
        #         break
        time.sleep(0.01)


In [ ]:

display(image_container)


try:
    Color_Recongnize()
except:
    pass


In [ ]:

cap.release()